# analysis_b / 01+03 — Posts-Only Pipeline

Combined data-preparation + LDA topic modelling, restricted to **posts only** (`type == 'post'`). Comments are excluded at load time.

**Reads:** `../reddit_scrapy/data/Paranormal_20250101_20251231_timerange.csv`  
**Writes:** `artifacts/posts_only/posts_clean.parquet`, `theta.npy`, `phi.npy`, `lda_vocab.json`, `lda_model.pkl`, `topic_labels.json`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import spacy
import re
import json
import textwrap
import warnings
import joblib
import itertools
from pathlib import Path
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.model_selection import train_test_split
from scipy.sparse import csc_matrix
from scipy.stats import spearmanr, rankdata

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

DATA_PATH  = Path('../reddit_scrapy/data/Paranormal_20250101_20251231_timerange.csv')
ARTIFACTS  = Path('artifacts/posts_only')
ARTIFACTS.mkdir(parents=True, exist_ok=True)

MIN_TOKENS = 5  # drop posts with fewer lemma tokens than this after spaCy

## 1. Load & Filter to Posts

In [ ]:
df_raw = pd.read_csv(DATA_PATH)
print(f'Raw shape : {df_raw.shape}')
print(f'type counts:\n{df_raw["type"].value_counts()}')

# Keep posts only — drop all comments
df_raw = df_raw[df_raw['type'] == 'post'].reset_index(drop=True)
print(f'\nAfter post filter: {df_raw.shape}')
print(f'type unique: {df_raw["type"].unique()}')
df_raw.head(3)

## 2. Column Standardization

In [ ]:
COLUMN_MAP = {
    'selftext'       : 'text',
    'body'           : 'text',
    'post_text'      : 'text',
    'score'          : 'score',
    'ups'            : 'score',
    'link_flair_text': 'flair',
    'flair'          : 'flair',
    'post_flair'     : 'flair',
    'created_utc'    : 'created_utc',
    'created'        : 'created_utc',
    'timestamp'      : 'created_utc',
    'id'             : 'post_id',
    'post_id'        : 'post_id',
    'title'          : 'title',
}

df = df_raw.rename(columns={k: v for k, v in COLUMN_MAP.items() if k in df_raw.columns})

required = ['text', 'created_utc']
missing  = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(
        f'Missing required columns after rename: {missing}.\n'
        f'Available columns: {df.columns.tolist()}\n'
        f'Update COLUMN_MAP above to match your CSV headers.'
    )

for col, default in [('score', np.nan), ('flair', 'unknown'), ('title', ''), ('post_id', None)]:
    if col not in df.columns:
        df[col] = default

print('Standardized columns:', df.columns.tolist())
print(f'Total rows: {len(df):,}')

## 3. Metadata Overview

In [ ]:
if df['score'].notna().any():
    cap = df['score'].quantile(0.99)
    fig, ax = plt.subplots(figsize=(6, 4))
    df['score'].clip(upper=cap).plot(kind='hist', bins=50, ax=ax)
    ax.set_title(f'Score distribution (clipped at 99th pct = {cap:.0f})')
    ax.set_xlabel('Score')
    plt.tight_layout()
    plt.show()
    print(df['score'].describe())
else:
    print('No score data available.')

## 4. Text Cleaning

Strip Reddit-specific markup. Original text preserved in `text_raw`.

In [ ]:
def clean_text(text):
    if not isinstance(text, str) or not text.strip():
        return ''
    if text.strip().lower() in ('[deleted]', '[removed]'):
        return ''
    text = re.sub(r'(?m)^>.*$', '', text)            # blockquotes
    text = re.sub(r'https?://\S+|www\.\S+', '', text) # URLs
    text = re.sub(r'\[([^\]]+)\]\([^\)]*\)', r'\1', text)  # markdown links
    text = re.sub(r'\*{1,3}([^*]+)\*{1,3}', r'\1', text)  # bold/italic
    text = re.sub(r'~~([^~]+)~~', r'\1', text)       # strikethrough
    text = re.sub(r'```[\s\S]*?```', '', text)        # fenced code
    text = re.sub(r'`[^`]+`', '', text)               # inline code
    text = re.sub(r'(?m)^#{1,6}\s', '', text)         # ATX headers
    text = re.sub(r'\bu/\w+', '', text)               # user mentions
    text = re.sub(r'\br/\w+', '', text)               # subreddit mentions
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

df['text_raw']   = df['text'].copy()
df['text_clean'] = df['text_raw'].apply(clean_text)

before = len(df)
df = df[df['text_clean'].str.len() > 0].reset_index(drop=True)
print(f'Dropped {before - len(df):,} empty/deleted posts. Remaining: {len(df):,}')

In [ ]:
# Most frequent authors
print(df['author'].value_counts().head(20))

In [ ]:
# Drop AutoModerator posts
before = len(df)
df = df[df['author'] != 'AutoModerator'].reset_index(drop=True)
print(f'Dropped {before - len(df):,} AutoModerator posts. Remaining: {len(df):,}')

## 5. spaCy Lemmatization

Pipeline: tokenize → POS-filter (NOUN, VERB, ADJ, ADV) → remove stopwords and short tokens → lemmatize. Posts below `MIN_TOKENS` after this step are dropped.

In [ ]:
pip install https://github.com/explosion/spacy-models/releases/download/en_core_web_lg-3.8.0/en_core_web_lg-3.8.0-py3-none-any.whl

In [ ]:
nlp = spacy.load('en_core_web_lg')
nlp.select_pipes(enable=['tok2vec', 'tagger', 'attribute_ruler', 'lemmatizer'])

KEEP_POS = {'NOUN', 'VERB', 'ADJ', 'ADV'}

def extract_tokens(doc):
    return [
        token.lemma_.lower()
        for token in doc
        if  token.pos_  in KEEP_POS
        and not token.is_stop
        and not token.is_punct
        and not token.like_num
        and not token.is_space
        and len(token.lemma_) > 2
    ]

In [ ]:
raw_word_count = df['text_clean'].str.split().apply(len)

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(raw_word_count, bins=80, edgecolor='black')
ax.axvline(MIN_TOKENS, color='red', linestyle='--', label=f'MIN_TOKENS = {MIN_TOKENS}')
ax.set_xlabel('Words per document (pre-lemmatization)')
ax.set_ylabel('Number of documents')
ax.set_title('Document length distribution (raw word count)')
ax.legend()
plt.tight_layout()
plt.show()

print(raw_word_count.describe())
print(f'\nDocs below MIN_TOKENS={MIN_TOKENS}: {(raw_word_count < MIN_TOKENS).sum():,} ({(raw_word_count < MIN_TOKENS).mean():.1%})')

In [ ]:
BATCH_SIZE = 64
texts      = df['text_clean'].tolist()

all_tokens = [
    extract_tokens(doc)
    for doc in nlp.pipe(texts, batch_size=BATCH_SIZE)
]

df['tokens']      = all_tokens
df['text_lemma']  = df['tokens'].apply(' '.join)
df['token_count'] = df['tokens'].apply(len)

before = len(df)
df = df[df['token_count'] >= MIN_TOKENS].reset_index(drop=True)
print(f'Dropped {before - len(df):,} posts below {MIN_TOKENS}-token floor. Remaining: {len(df):,}')
print(f'Median token count: {df["token_count"].median():.0f}')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
df['token_count'].clip(upper=500).plot(kind='hist', bins=60, ax=ax)
ax.set_title('Token count per post after lemmatization (clipped at 500)')
ax.set_xlabel('Tokens')
plt.tight_layout()
plt.show()

## 6. Temporal Features

Parses `created_utc`, extracts `year_month`, `year`, `month`, and **`date`** (calendar day) and **`day_idx`** (integer 0 → N−1).

In [ ]:
df['created_utc'] = pd.to_datetime(df['created_utc'], errors='coerce')

n_bad = df['created_utc'].isnull().sum()
if n_bad > 0:
    print(f'Warning: {n_bad} unparseable timestamps — dropping.')
    df = df.dropna(subset=['created_utc']).reset_index(drop=True)

df['year_month'] = df['created_utc'].dt.to_period('M').astype(str)
df['year']       = df['created_utc'].dt.year
df['month']      = df['created_utc'].dt.month
df['date']       = df['created_utc'].dt.normalize()
df['day_idx']    = (df['date'] - df['date'].min()).dt.days

print(f'Date range : {df["date"].min().date()} → {df["date"].max().date()}')
print(f'Span       : {int(df["day_idx"].max()) + 1} days')
print('Months present:', sorted(df['year_month'].unique()))

In [ ]:
daily_counts = df.groupby('date').size().sort_index()

fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(range(len(daily_counts)), daily_counts.values, color='steelblue', width=0.8)
ax.set_xticks(range(len(daily_counts)))
ax.set_xticklabels(
    [d.strftime('%b %d') for d in daily_counts.index], rotation=45, ha='right'
)
ax.set_title('Posts per day (posts only, after cleaning + token filter)')
ax.set_xlabel('Date')
ax.set_ylabel('Posts')
plt.tight_layout()
plt.show()

## 7. Write Artifact

Writes `artifacts/posts_only/posts_clean.parquet`.

| Column | Type | Notes |
|---|---|---|
| `post_id` | str | Reddit post ID |
| `post_title` | str | Post title |
| `text_raw` | str | Original scraped text |
| `text_clean` | str | Markup-stripped text |
| `text_lemma` | str | Space-joined lemmas |
| `tokens` | list[str] | Lemma token list |
| `token_count` | int | Post length in tokens |
| `score` | float | Upvote score |
| `flair` | str | Post flair |
| `created_utc` | datetime | UTC post time |
| `year_month` | str | e.g. `'2025-01'` |
| `year` | int | |
| `month` | int | |
| `date` | datetime | Calendar day (midnight) |
| `day_idx` | int | Days since first post (0-based) |

In [ ]:
SAVE_COLS = [
    'post_id', 'type', 'post_title', 'text_raw', 'text_clean', 'text_lemma',
    'tokens', 'token_count', 'score', 'flair',
    'created_utc', 'year_month', 'year', 'month', 'date', 'day_idx',
]
SAVE_COLS = [c for c in SAVE_COLS if c in df.columns]

out_path = ARTIFACTS / 'posts_clean.parquet'
df[SAVE_COLS].to_parquet(out_path, index=False)

print(f'Saved {len(df):,} posts → {out_path}')
print(f'Columns : {SAVE_COLS}')
print(f'File size: {out_path.stat().st_size / 1024:.1f} KB')

---
# Part 2 — Latent Topic Discovery (LDA)

Unsupervised discovery of latent thematic structure in the posts-only corpus. No pre-imposed categories — topics emerge from the data.

**Pipeline:** DTM → K selection (perplexity + PMI coherence) → fit final LDA → topic profiling (FREX + top documents) → score correlations

## 8. Document-Term Matrix

In [ ]:
vec = CountVectorizer(min_df=10, max_df=0.90)
dtm = vec.fit_transform(df['text_lemma'])
vocab = vec.get_feature_names_out()

print(f'DTM      : {dtm.shape[0]:,} docs × {dtm.shape[1]:,} terms')
print(f'Sparsity : {1 - dtm.nnz / (dtm.shape[0] * dtm.shape[1]):.2%}')

X_train, X_test = train_test_split(dtm, test_size=0.10, random_state=42)
print(f'Train: {X_train.shape[0]:,}   Test: {X_test.shape[0]:,}')

## 9. K Selection

- **Perplexity** on held-out 10% — lower is better; tends to keep decreasing, so not sufficient alone
- **PMI coherence** — higher means more semantically coherent topics; choose K at the elbow

In [ ]:
dtm_csc = csc_matrix((dtm > 0).astype(np.float32))

def pmi_coherence(topic_idx, phi, dtm_csc, n_top=10):
    top_w = phi[topic_idx].argsort()[::-1][:n_top]
    cols  = [
        np.asarray(dtm_csc.getcol(int(w)).todense()).flatten() > 0
        for w in top_w
    ]
    pmis = []
    for i in range(len(cols)):
        for j in range(i + 1, len(cols)):
            p_i  = cols[i].mean()
            p_j  = cols[j].mean()
            p_ij = (cols[i] & cols[j]).mean()
            if p_i > 0 and p_j > 0 and p_ij > 0:
                pmis.append(np.log(p_ij / (p_i * p_j)))
    return float(np.mean(pmis)) if pmis else 0.0

In [ ]:
K_RANGE = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
results = []

for K in K_RANGE:
    lda_k = LatentDirichletAllocation(
        n_components=K, learning_method='online', max_iter=20, random_state=42
    )
    lda_k.fit(X_train)
    perp  = lda_k.perplexity(X_test)
    phi_k = lda_k.components_ / lda_k.components_.sum(axis=1, keepdims=True)
    coh   = float(np.mean([pmi_coherence(k, phi_k, dtm_csc) for k in range(K)]))
    results.append({'K': K, 'perplexity': perp, 'coherence': coh})
    print(f'K={K:2d}  perplexity={perp:9.1f}  coherence={coh:.4f}')

results_df = pd.DataFrame(results)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.plot(results_df['K'], results_df['perplexity'], marker='o')
ax1.set_title('Perplexity on held-out 10%')
ax1.set_xlabel('K')
ax1.set_ylabel('Perplexity (lower = better)')

ax2.plot(results_df['K'], results_df['coherence'], marker='o', color='steelblue')
ax2.set_title('Mean PMI Coherence')
ax2.set_xlabel('K')
ax2.set_ylabel('Mean pairwise PMI (higher = better)')

plt.suptitle('K selection — choose K at the coherence elbow')
plt.tight_layout()
plt.show()

## 10. Fit Final Model

Set `K_FINAL` based on the plot above.

In [ ]:
K_FINAL = 4

lda = LatentDirichletAllocation(
    n_components=K_FINAL, learning_method='batch', max_iter=100, random_state=42
)
lda.fit(dtm)

print(f'Final model: K={K_FINAL}, held-out perplexity={lda.perplexity(X_test):.1f}')

In [ ]:
# phi   : (K, V) normalized topic-word matrix
# theta : (N, K) document-topic distribution
phi   = lda.components_ / lda.components_.sum(axis=1, keepdims=True)
theta = lda.transform(dtm)

print(f'phi   shape : {phi.shape}')
print(f'theta shape : {theta.shape}')
print(f'theta row-sum check (should be ~1.0): {theta.sum(axis=1).mean():.4f}')

## 11. Topic Profiling

For each topic: top words + top documents by theta loading. Read the documents — do not rely on words alone.

In [ ]:
N_TOP_WORDS = 20
vocab_list  = list(vocab)

print('Top words per topic')
print('=' * 70)
for k in range(K_FINAL):
    top_idx   = phi[k].argsort()[::-1][:N_TOP_WORDS]
    top_words = ', '.join(vocab_list[i] for i in top_idx)
    print(f'Topic {k:2d}: {top_words}')

In [ ]:
N_TOP = 20

top_word_sets = {}
for k in range(K_FINAL):
    top_idx = phi[k].argsort()[::-1][:N_TOP]
    top_word_sets[k] = set(vocab_list[i] for i in top_idx)

overlap = np.zeros((K_FINAL, K_FINAL), dtype=int)
for i, j in itertools.combinations(range(K_FINAL), 2):
    shared = top_word_sets[i] & top_word_sets[j]
    overlap[i, j] = len(shared)
    overlap[j, i] = len(shared)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    overlap, annot=True, fmt='d', cmap='YlOrRd',
    xticklabels=[f'T{k}' for k in range(K_FINAL)],
    yticklabels=[f'T{k}' for k in range(K_FINAL)],
    ax=ax
)
ax.set_title(f'Shared top-{N_TOP} words between topic pairs')
plt.tight_layout()
plt.show()

print(f'\nShared words (top-{N_TOP}) per pair:')
for i, j in itertools.combinations(range(K_FINAL), 2):
    shared = sorted(top_word_sets[i] & top_word_sets[j])
    print(f'  T{i} ∩ T{j} ({len(shared)}): {", ".join(shared) if shared else "none"}')

### FREX — Frequency × Exclusivity

Top words by raw probability alone are dominated by corpus-wide filler (*think*, *see*, *people*).
FREX re-ranks words by the harmonic mean of two ECDF-normalised scores:

- **Frequency** `phi[k, w]` — probability of the word within the topic  
- **Exclusivity** `phi[k, w] / Σ_k phi[k, w]` — share of the word's total mass in this topic

A word scores high only when it is *both* common in the topic *and* concentrated there.

In [ ]:
def frex(phi, w=0.5, n_top=15):
    K, V = phi.shape
    col_sums = phi.sum(axis=0)
    excl = phi / np.where(col_sums > 0, col_sums, 1)
    freq_ecdf = np.apply_along_axis(
        lambda row: rankdata(row) / V, axis=1, arr=phi
    )
    excl_ecdf = np.apply_along_axis(
        lambda row: rankdata(row) / V, axis=1, arr=excl
    )
    scores = 1.0 / (w / freq_ecdf + (1 - w) / excl_ecdf)
    top_idx = scores.argsort(axis=1)[:, ::-1][:, :n_top]
    return scores, top_idx


N_FREX   = 15
FREX_W   = 0.5

frex_scores, frex_top = frex(phi, w=FREX_W, n_top=N_FREX)

print(f'FREX top-{N_FREX} words per topic  (w={FREX_W})')
print('=' * 70)
for k in range(K_FINAL):
    words = ', '.join(vocab_list[i] for i in frex_top[k])
    print(f'Topic {k}: {words}')

In [ ]:
N_CMP = 10
cols = pd.MultiIndex.from_product(
    [[f'Topic {k}' for k in range(K_FINAL)], ['Raw', 'FREX']]
)
data = {}
for k in range(K_FINAL):
    raw_words  = [vocab_list[i] for i in phi[k].argsort()[::-1][:N_CMP]]
    frex_words = [vocab_list[i] for i in frex_top[k][:N_CMP]]
    data[(f'Topic {k}', 'Raw')]  = raw_words
    data[(f'Topic {k}', 'FREX')] = frex_words

cmp_df = pd.DataFrame(data, index=range(1, N_CMP + 1))
cmp_df.index.name = 'Rank'
cmp_df

### Doc Inspection

In [ ]:
N_TOP_DOCS = 10

for k in range(K_FINAL):
    print('=' * 70)
    print(f'TOPIC {k} — top {N_TOP_DOCS} documents')
    print('=' * 70)
    top_doc_idx = theta[:, k].argsort()[::-1][:N_TOP_DOCS]
    for rank, idx in enumerate(top_doc_idx, start=1):
        row     = df.iloc[idx]
        loading = theta[idx, k]
        snippet = row['text_clean']
        date    = str(row.get('date', row.get('year_month', '')))
        print(f'  [{rank}] theta={loading:.3f} | {date}')
        print(textwrap.fill(snippet, width=76, initial_indent='      ', subsequent_indent='      '))
        print()
    print()

In [ ]:
N_TOP_DOCS = 10
md_path    = ARTIFACTS / 'top_docs.md'

lines = []
for k in range(K_FINAL):
    lines.append(f'# TOPIC {k} — top {N_TOP_DOCS} documents\n\n')
    top_doc_idx = theta[:, k].argsort()[::-1][:N_TOP_DOCS]
    for rank, idx in enumerate(top_doc_idx, start=1):
        row     = df.iloc[idx]
        loading = theta[idx, k]
        snippet = str(row['text_clean'])
        date    = str(row.get('date', row.get('year_month', '')))
        lines.append(f'- [{rank}] theta={loading:.3f} | {date}\n\n')
        wrapped = textwrap.fill(snippet, width=76,
                                initial_indent='  ',
                                subsequent_indent='  ')
        lines.append(f'{wrapped}\n\n')
    lines.append('\n---\n\n')

md_path.write_text(''.join(lines), encoding='utf-8')
print(f'Written → {md_path}')

In [ ]:
N_SAMPLE    = 5
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

W = 82

dominant = theta.argmax(axis=1)

for k in range(K_FINAL):
    print('\n' + '─' * W)
    print(f'  TOPIC {k} — {N_SAMPLE} sampled posts (dominant-member only)')
    print('─' * W)

    members    = np.where(dominant == k)[0]
    sample_idx = rng.choice(members, size=min(N_SAMPLE, len(members)), replace=False)

    for rank, idx in enumerate(sample_idx, start=1):
        row     = df.iloc[idx]
        loading = theta[idx, k]
        date    = str(row.get('date', row.get('year_month', '')))

        print(f'\n  [{rank}]  {date}  |  score={row["score"]}  |  theta={loading:.3f}')
        print('  ' + '─' * (W - 2))

        paragraph = textwrap.fill(str(row['text_clean']), width=W - 4,
                                  initial_indent='  ',
                                  subsequent_indent='  ')
        print(paragraph)
        print()

## 12. Topic Distribution Heatmap

Mean theta per topic across the full corpus — shows which topics dominate overall.

In [ ]:
mean_theta = theta.mean(axis=0)
topic_labels_tmp = [f'Topic {k}' for k in range(K_FINAL)]

fig, ax = plt.subplots(figsize=(10, 3))
ax.bar(range(K_FINAL), mean_theta, color='steelblue')
ax.set_xticks(range(K_FINAL))
ax.set_xticklabels(topic_labels_tmp, rotation=45, ha='right')
ax.set_title('Mean theta loading per topic (corpus-wide, posts only)')
ax.set_ylabel('Mean theta')
plt.tight_layout()
plt.show()

## 13. Score Correlation

Spearman rho between post score and each topic's theta loading. Bonferroni-corrected.

In [ ]:
score_vals  = df['score'].fillna(0).values
alpha_bonf  = 0.05 / K_FINAL

print('Spearman rho: score ~ topic theta loading')
print(f'(* = Bonferroni-significant, threshold p < {alpha_bonf:.4f})')
print()
for k in range(K_FINAL):
    rho, p = spearmanr(score_vals, theta[:, k])
    sig    = '*' if p < alpha_bonf else ''
    print(f'  Topic {k:2d}: rho={rho:+.3f}  p={p:.4f}  {sig}')

## 14. Write LDA Artifacts

In [ ]:
np.save(ARTIFACTS / 'theta.npy', theta)
np.save(ARTIFACTS / 'phi.npy',   phi)

with open(ARTIFACTS / 'lda_vocab.json', 'w') as fh:
    json.dump(list(vocab), fh)

joblib.dump(lda, ARTIFACTS / 'lda_model.pkl')

template = {str(k): f'Topic {k} — label here' for k in range(K_FINAL)}
with open(ARTIFACTS / 'topic_labels.json', 'w') as fh:
    json.dump(template, fh, indent=2)

print('Artifacts written:')
for name in ('posts_clean.parquet', 'theta.npy', 'phi.npy',
             'lda_vocab.json', 'lda_model.pkl', 'topic_labels.json', 'top_docs.md'):
    p = ARTIFACTS / name
    if p.exists():
        print(f'  {p}  ({p.stat().st_size / 1024:.1f} KB)')

print()
print('Fill in topic_labels.json before running downstream notebooks.')